# 🍳 Recipe 03 — RAG App (PDF Q&A)

> **AI Cookbook** by Poorvi Bajpai

---

## 🎯 What you'll learn
- What RAG (Retrieval-Augmented Generation) is and why it matters
- How to load and chunk a document
- How to create a vector store with FAISS + HuggingFace Embeddings
- How to build a retrieval chain with LangChain + Groq
- How to ask questions and get grounded answers from any document

**Stack:** LangChain · FAISS · HuggingFace Embeddings · Groq LLM

---

## 🧠 What is RAG?

**RAG = Retrieval-Augmented Generation**

Instead of relying on what an LLM memorized during training, RAG first **retrieves** relevant context from your own documents, then passes it to the LLM to **generate** a grounded answer.


User Question
↓
[Embed question] → Search FAISS → Retrieve top-k chunks
↓
[LLM] gets: question + chunks → Grounded answer


**Why it matters:**
- LLMs hallucinate less when given real context
- Works on private documents — no fine-tuning needed
- Powers most enterprise AI products today

In [ ]:
# Uncomment and run once
#!pip install langchain langchain-community langchain-groq faiss-cpu pypdf sentence-transformers python-dotenv

In [15]:
import sys
!{sys.executable} -m pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface langchain-faiss faiss-cpu
!{sys.executable} -m pip install --no-cache-dir langchain==0.3.25 langchain-core==0.3.60 langchain-community==0.3.22 langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers

Found existing installation: langchain 1.2.17
Uninstalling langchain-1.2.17:
  Successfully uninstalled langchain-1.2.17
Found existing installation: langchain-core 1.4.7
Uninstalling langchain-core-1.4.7:
  Successfully uninstalled langchain-core-1.4.7
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langchain-groq 1.1.2
Uninstalling langchain-groq-1.1.2:
  Successfully uninstalled langchain-groq-1.1.2
Found existing installation: langchain-huggingface 1.2.2
Uninstalling langchain-huggingface-1.2.2:
  Successfully uninstalled langchain-huggingface-1.2.2
Found existing installation: langchain-faiss 0.1.1
Uninstalling langchain-faiss-0.1.1:
  Successfully uninstalled langchain-faiss-0.1.1
Found existi

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.60 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-google-genai 4.2.2 requires langchain-core<2.0.0,>=1.2.29, but you have langchain-core 0.3.60 which is incompatible.
langgraph 1.1.10 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.60 which is incompatible.
langgraph-prebuilt 1.0.13 requires langchain-core>=1.3.1, but you have langchain-core 0.3.60 which is incompatible.


In [17]:
import importlib.metadata
for pkg in ["langchain", "langchain-core", "langchain-community"]:
    print(pkg, importlib.metadata.version(pkg))

langchain 0.3.25
langchain-core 0.3.60
langchain-community 0.3.22


In [6]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain_core.documents import Document

print("✅ All imports successful!")

✅ All imports successful!


In [7]:
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("✅ API Key loaded!" if GROQ_API_KEY else "❌ API Key not found — check your .env file")

✅ API Key loaded!


---
## 📌 Part 1 — Create Sample Knowledge Base

We create a text-based knowledge base for demo purposes.
Swap with any PDF later using PyPDFLoader.

In [8]:
sample_text = """
Artificial Intelligence: A Brief Overview

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines 
that are programmed to think and learn like humans. The term was coined by John McCarthy 
in 1956 at the Dartmouth Conference, which is considered the birth of AI as a field.

Machine Learning is a subset of AI that enables computers to learn from data without 
being explicitly programmed. Key types include supervised learning, unsupervised learning, 
and reinforcement learning.

Deep Learning is a subset of machine learning that uses neural networks with many layers. 
It has revolutionized fields like computer vision, natural language processing, and speech 
recognition. Key architectures include CNNs, RNNs, and Transformers.

Large Language Models (LLMs) are deep learning models trained on massive text datasets. 
Examples include GPT-4, Claude, Gemini, and LLaMA. They can generate text, answer 
questions, summarize documents, and write code.

RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval 
with text generation. It allows LLMs to access external knowledge bases, reducing 
hallucination and enabling use of private or up-to-date information.

Vector databases store embeddings — numerical representations of text — and enable 
semantic similarity search. Popular options include FAISS, Pinecone, ChromaDB, and Weaviate.

Prompt Engineering is the practice of designing inputs to LLMs to get desired outputs. 
Techniques include zero-shot prompting, few-shot prompting, chain-of-thought, and 
retrieval augmentation.
"""

with open("ai_knowledge_base.txt", "w") as f:
    f.write(sample_text)

print(f"✅ Knowledge base created! Total characters: {len(sample_text)}")

✅ Knowledge base created! Total characters: 1605


---
## 📌 Part 2 — Load & Chunk the Document

Split into overlapping chunks so retrieval finds relevant pieces without losing context at boundaries.

In [9]:
with open("ai_knowledge_base.txt", "r") as f:
    text = f.read()

docs = [Document(page_content=text, metadata={"source": "ai_knowledge_base.txt"})]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(docs)

print(f"✅ Document split into {len(chunks)} chunks")
print(f"\nSample chunk:\n{'-'*50}")
print(chunks[0].page_content)
print("-"*50)

✅ Document split into 4 chunks

Sample chunk:
--------------------------------------------------
Artificial Intelligence: A Brief Overview

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines 
that are programmed to think and learn like humans. The term was coined by John McCarthy 
in 1956 at the Dartmouth Conference, which is considered the birth of AI as a field.
--------------------------------------------------


---
## 📌 Part 3 — Create Embeddings & Vector Store

Convert each chunk into a vector using HuggingFace, store in FAISS for fast similarity search.

In [ ]:
print("Loading embedding model... (first run downloads ~90MB)")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded!")

print("\nCreating vector store...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store created with {vectorstore.index.ntotal} vectors!")

Loading embedding model... (first run downloads ~90MB)


C:\Users\poorv\AppData\Local\Temp\ipykernel_12980\449972898.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\poorv\anaconda3\envs\ai-cookbook\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\poorv\anaconda3\envs\ai-cookbook\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C

In [ ]:
# Test retrieval
query = "What is RAG and why is it useful?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
relevant_docs = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(relevant_docs):
    print(f"Chunk {i+1}:")
    print(doc.page_content[:200] + "...")
    print()

---
## 📌 Part 4 — Set Up LLM + RetrievalQA Chain

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=GROQ_API_KEY
)

prompt_template = """You are a helpful AI assistant. Use the following context to answer 
the question accurately. If the answer is not in the context, say "I don't have enough 
information in the provided document to answer this."

Context:
{context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

print("✅ RAG chain ready!")

---
## 📌 Part 5 — Ask Questions!

In [ ]:
def ask(question):
    """Ask a question and get a grounded answer from the document."""
    result = qa_chain.invoke({"query": question})
    print(f"❓ Question: {question}")
    print(f"\n💡 Answer: {result['result']}")
    print(f"\n📄 Sources used: {len(result['source_documents'])} chunks")
    print("=" * 70)

In [ ]:
questions = [
    "What is RAG and why is it useful?",
    "Who coined the term Artificial Intelligence and when?",
    "What are the types of machine learning?",
    "What are some examples of Large Language Models?",
    "What is a vector database? Give some examples.",
]

for q in questions:
    ask(q)
    print()

In [ ]:
# Test out-of-scope — model should say it doesn't know
ask("What is the capital of France?")

---
## 📌 Part 6 — Upgrade: Use a Real PDF

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("your_document.pdf")
docs = loader.load()
chunks = splitter.split_documents(docs)
```

Any PDF works — research paper, resume, textbook, company docs.

---
## 📝 Key Takeaways

- RAG = Retrieve chunks → Pass as context → LLM generates grounded answer
- `all-MiniLM-L6-v2` is fast, lightweight, great for local embeddings
- FAISS stores vectors locally — no external database needed
- `temperature=0` makes Q&A deterministic and reliable
- This is the most in-demand AI architecture in production today

---
**Next Recipe →** `04-ai-agent/notebook.ipynb`